# 美赛 C 题数据预处理：Dancing with the Stars 分析

## 题目背景

**Dancing with the Stars (DWTS)** 是一档美国真人秀节目，名人与专业舞者配对后每周进行舞蹈表演。评委打分与观众投票结合决定淘汰结果。

### 核心任务
1. **估算观众投票**：开发数学模型估算未知的观众投票数据
2. **对比投票方法**：分析「排名法」vs「百分比法」两种计票方式的差异
3. **争议案例分析**：分析特定选手（Jerry Rice、Bobby Bones等）的争议结果
4. **影响因素建模**：分析舞伴、选手特征对成绩的影响
5. **改进建议**：提出更公平的投票机制

### 数据说明
- **数据来源**：2026_MCM_Problem_C_Data.csv
- **覆盖范围**：第1-34季全部选手数据
- **核心字段**：选手信息、评委打分、比赛结果

---
## 第一步：数据加载与初探

首先导入必要的库并加载数据。我们使用 Polars 进行高效数据处理，同时保留 Pandas 兼容性以便可视化。

In [1]:
# 导入数据处理库
import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path

# 设置显示选项
pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(50)

# 定义路径
DATA_RAW = Path("data/raw")
DATA_PROCESSED = Path("data/processed")
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print("✅ 库导入完成")

✅ 库导入完成


### 1.1 加载原始数据

数据文件包含34季的选手信息和评委打分。由于存在 "N/A" 文本值，需要在加载时特殊处理。

In [2]:
# 加载CSV数据，将 "N/A" 识别为空值
df_raw = pl.read_csv(
    DATA_RAW / "2026_MCM_Problem_C_Data.csv",
    null_values=["N/A", "NA", ""],
    infer_schema_length=10000  # 增加推断行数以提高类型识别准确性
)

print(f"📊 数据维度: {df_raw.shape[0]} 行 × {df_raw.shape[1]} 列")
print(f"📅 覆盖季数: 第 {df_raw['season'].min()} 季 ~ 第 {df_raw['season'].max()} 季")
print(f"👥 总选手数: {df_raw.shape[0]} 人")

📊 数据维度: 421 行 × 53 列
📅 覆盖季数: 第 1 季 ~ 第 34 季
👥 总选手数: 421 人


### 1.2 查看数据结构

使用 `.schema` 和 `.head()` 了解数据的列名、数据类型和前几行内容。

In [3]:
# 查看数据模式（列名和类型）
print("📋 数据列结构:")
for col_name, col_type in df_raw.schema.items():
    print(f"  {col_name}: {col_type}")

📋 数据列结构:
  celebrity_name: String
  ballroom_partner: String
  celebrity_industry: String
  celebrity_homestate: String
  celebrity_homecountry/region: String
  celebrity_age_during_season: Int64
  season: Int64
  results: String
  placement: Int64
  week1_judge1_score: Float64
  week1_judge2_score: Float64
  week1_judge3_score: Float64
  week1_judge4_score: Int64
  week2_judge1_score: Float64
  week2_judge2_score: Float64
  week2_judge3_score: Float64
  week2_judge4_score: Float64
  week3_judge1_score: Float64
  week3_judge2_score: Float64
  week3_judge3_score: Float64
  week3_judge4_score: Float64
  week4_judge1_score: Float64
  week4_judge2_score: Float64
  week4_judge3_score: Float64
  week4_judge4_score: Float64
  week5_judge1_score: Float64
  week5_judge2_score: Float64
  week5_judge3_score: Float64
  week5_judge4_score: Float64
  week6_judge1_score: Float64
  week6_judge2_score: Float64
  week6_judge3_score: Float64
  week6_judge4_score: Float64
  week7_judge1_score: Float64
  w

In [4]:
# 查看前10行数据
df_raw.head(10)

celebrity_name,ballroom_partner,celebrity_industry,celebrity_homestate,celebrity_homecountry/region,celebrity_age_during_season,season,results,placement,week1_judge1_score,week1_judge2_score,week1_judge3_score,week1_judge4_score,week2_judge1_score,week2_judge2_score,week2_judge3_score,week2_judge4_score,week3_judge1_score,week3_judge2_score,week3_judge3_score,week3_judge4_score,week4_judge1_score,week4_judge2_score,week4_judge3_score,week4_judge4_score,week5_judge1_score,week5_judge2_score,week5_judge3_score,week5_judge4_score,week6_judge1_score,week6_judge2_score,week6_judge3_score,week6_judge4_score,week7_judge1_score,week7_judge2_score,week7_judge3_score,week7_judge4_score,week8_judge1_score,week8_judge2_score,week8_judge3_score,week8_judge4_score,week9_judge1_score,week9_judge2_score,week9_judge3_score,week9_judge4_score,week10_judge1_score,week10_judge2_score,week10_judge3_score,week10_judge4_score,week11_judge1_score,week11_judge2_score,week11_judge3_score,week11_judge4_score
str,str,str,str,str,i64,i64,str,i64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""John O'Hurley""","""Charlotte Jorgensen""","""Actor/Actress""","""Maine""","""United States""",50,1,"""2nd Place""",2,7.0,7.0,6.0,null,8.0,9.0,9.0,null,9.0,8.0,7.0,null,7.0,8.0,6.0,null,9.0,9.0,9.0,null,9.0,9.0,9.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""Kelly Monaco""","""Alec Mazo""","""Actor/Actress""","""Pennsylvania""","""United States""",29,1,"""1st Place""",1,5.0,4.0,4.0,null,5.0,6.0,6.0,null,6.0,7.0,8.0,null,9.0,9.0,8.0,null,8.5,7.5,7.5,null,8.5,9.5,9.5,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""Evander Holyfield""","""Edyta Sliwinska""","""Athlete""","""Alabama""","""United States""",42,1,"""Eliminated Week 3""",5,5.0,7.0,6.0,null,5.0,4.0,5.0,null,5.0,4.0,4.0,null,0.0,0.0,0.0,null,0.0,0.0,0.0,null,0.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""Rachel Hunter""","""Jonathan Roberts""","""Model""",null,"""New Zealand""",35,1,"""Eliminated Week 4""",4,7.0,6.0,7.0,null,8.0,8.0,8.0,null,8.0,9.0,9.0,null,7.0,9.0,9.0,null,0.0,0.0,0.0,null,0.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""Joey McIntyre""","""Ashly DelGrosso""","""Singer/Rapper""","""Massachusetts""","""United States""",32,1,"""3rd Place""",3,7.0,7.0,6.0,null,8.0,7.0,6.0,null,7.0,7.0,8.0,null,7.0,6.0,7.0,null,8.5,7.0,7.0,null,0.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""Trista Sutter""","""Louis van Amstel""","""TV Personality""","""Indiana""","""United States""",32,1,"""Eliminated Week 2""",6,6.0,6.0,6.0,null,6.0,7.0,6.0,null,0.0,0.0,0.0,null,0.0,0.0,0.0,null,0.0,0.0,0.0,null,0.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""Tatum O'Neal""","""Nick Kosovich""","""Actor/Actress""","""California""","""United States""",42,2,"""Eliminated Week 2""",9,7.0,8.0,8.0,null,5.0,6.0,6.0,null,0.0,0.0,0.0,null,0.0,0.0,0.0,null,0.0,0.0,0.0,null,0.0,0.0,0.0,null,0.0,0.0,0.0,null,0.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null
"""Tia Carrere""","""Maksim Chmerkoskiy""","""Actor/Actress""","""Hawaii""","""United States""",39,2,"""Eliminated Week 5""",6,6.0,7.0,7.0,null,7.0,8.0,7.0,null,9.0,8.0,9.0,null,9.0,8.0,8.0,null,7.0,7.0,8.0,null,0.0,0.0,0.0,null,0.0,0.0,0.0,null,0.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null
"""George Hamilton""","""Edyta Sliwinska""","""Actor/Actress""","""Tennessee""","""United States""",66,2,"""Eliminated Week 6""",5,7.0,5.0,6.0,null,8.0,7.0,7.0,null,7.0,7.0,8.0,null,7.0,7.0,7.0,null,8.0,8.0,

### 1.3 数据特点总结

通过观察原始数据，我们发现以下关键特点：

| 特点 | 说明 |
|------|------|
| **宽表结构** | 每周评分按列存储（week1_judge1_score ~ week11_judge4_score），共44列评分 |
| **N/A 值含义多重** | ①第4位评委不存在 ②选手已淘汰 ③该季周数不足 |
| **0分含义** | 选手在该周已被淘汰 |
| **小数分数** | 多舞平均分或奖励分平摊 |
| **特殊结果** | 存在 "Withdrew"（退赛）情况 |

In [5]:
# 统计基本信息
print("=" * 50)
print("📊 基础统计信息")
print("=" * 50)

# 各列缺失值统计
null_counts = df_raw.null_count()
print("\n🔍 缺失值统计（前15列）:")
for col in list(df_raw.columns)[:15]:
    null_val = null_counts[col][0]
    pct = null_val / len(df_raw) * 100
    print(f"  {col}: {null_val} ({pct:.1f}%)")

📊 基础统计信息

🔍 缺失值统计（前15列）:
  celebrity_name: 0 (0.0%)
  ballroom_partner: 0 (0.0%)
  celebrity_industry: 0 (0.0%)
  celebrity_homestate: 56 (13.3%)
  celebrity_homecountry/region: 0 (0.0%)
  celebrity_age_during_season: 0 (0.0%)
  season: 0 (0.0%)
  results: 0 (0.0%)
  placement: 0 (0.0%)
  week1_judge1_score: 0 (0.0%)
  week1_judge2_score: 0 (0.0%)
  week1_judge3_score: 14 (3.3%)
  week1_judge4_score: 340 (80.8%)
  week2_judge1_score: 0 (0.0%)
  week2_judge2_score: 0 (0.0%)


In [ ]:
# 结果类型分布
print("\n🏆 比赛结果分布:")
results_dist = df_raw.group_by("results").agg(pl.len().alias("count")).sort("count", descending=True)
print(results_dist)


🏆 比赛结果分布:
shape: (17, 2)
┌────────────────────┬───────┐
│ results            ┆ count │
│ ---                ┆ ---   │
│ str                ┆ u32   │
╞════════════════════╪═══════╡
│ Eliminated Week 2  ┆ 40    │
│ Eliminated Week 8  ┆ 36    │
│ Eliminated Week 7  ┆ 36    │
│ Eliminated Week 4  ┆ 35    │
│ 2nd Place          ┆ 34    │
│ 1st Place          ┆ 34    │
│ 3rd Place          ┆ 34    │
│ Eliminated Week 3  ┆ 33    │
│ Eliminated Week 9  ┆ 32    │
│ Eliminated Week 6  ┆ 31    │
│ Eliminated Week 5  ┆ 24    │
│ Eliminated Week 1  ┆ 16    │
│ Eliminated Week 10 ┆ 11    │
│ Withdrew           ┆ 10    │
│ 4th Place          ┆ 8     │
│ Eliminated Week 11 ┆ 4     │
│ 5th Place          ┆ 3     │
└────────────────────┴───────┘


C:\Users\fesmoph\AppData\Local\Temp\ipykernel_35200\1443837561.py:3: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  results_dist = df_raw.group_by("results").agg(pl.count().alias("count")).sort("count", descending=True)


In [7]:
# 选手职业分布
print("\n🎭 选手职业分布:")
industry_dist = df_raw.group_by("celebrity_industry").agg(pl.len().alias("count")).sort("count", descending=True)
print(industry_dist)


🎭 选手职业分布:
shape: (26, 2)
┌──────────────────────────┬───────┐
│ celebrity_industry       ┆ count │
│ ---                      ┆ ---   │
│ str                      ┆ u32   │
╞══════════════════════════╪═══════╡
│ Actor/Actress            ┆ 128   │
│ Athlete                  ┆ 95    │
│ TV Personality           ┆ 67    │
│ Singer/Rapper            ┆ 61    │
│ Model                    ┆ 17    │
│ Comedian                 ┆ 12    │
│ Social Media Personality ┆ 8     │
│ Radio Personality        ┆ 4     │
│ Entrepreneur             ┆ 4     │
│ Racing Driver            ┆ 4     │
│ …                        ┆ …     │
│ Military                 ┆ 1     │
│ Social media personality ┆ 1     │
│ Beauty Pagent            ┆ 1     │
│ Fitness Instructor       ┆ 1     │
│ Conservationist          ┆ 1     │
│ Astronaut                ┆ 1     │
│ Motivational Speaker     ┆ 1     │
│ Fashion Designer         ┆ 1     │
│ Producer                 ┆ 1     │
│ Magician                 ┆ 1     │
└───────────

In [ ]:
# 每季选手数量
print("\n📅 每季选手数量:")
season_counts = df_raw.group_by("season").agg(pl.len().alias("contestants")).sort("season")
print(season_counts)

---
## 第二步：数据清洗

根据数据特点，我们需要进行以下清洗操作。

### 2.1 缺失值处理策略

**核心原则**：N/A 值在本数据集中具有明确的业务含义，不应简单删除或填充。

| 缺失类型 | 处理策略 | 原因 |
|----------|----------|------|
| `week*_judge4_score` 为 N/A | 保留空值 | 该周只有3位评委 |
| 后期周数为 N/A | 保留空值 | 该季比赛周数较短 |
| 淘汰后分数为 0 | 在分析时过滤 | 0分不代表实际表现 |
| `celebrity_homestate` 为空 | 保留空值 | 选手非美国人 |

In [8]:
# 创建工作副本
df = df_raw.clone()

# 检查关键列的缺失情况
key_cols = ["celebrity_name", "ballroom_partner", "celebrity_industry", "season", "results", "placement"]
print("🔍 关键列缺失检查:")
for col in key_cols:
    null_count = df.filter(pl.col(col).is_null()).shape[0]
    print(f"  {col}: {null_count} 个缺失值")

# 检查 homestate 缺失（非美国选手）
non_us = df.filter(pl.col("celebrity_homestate").is_null())
print(f"\n🌍 非美国选手（homestate 为空）: {non_us.shape[0]} 人")
print(non_us.select(["celebrity_name", "celebrity_homecountry/region"]).head(10))

🔍 关键列缺失检查:
  celebrity_name: 0 个缺失值
  ballroom_partner: 0 个缺失值
  celebrity_industry: 0 个缺失值
  season: 0 个缺失值
  results: 0 个缺失值
  placement: 0 个缺失值

🌍 非美国选手（homestate 为空）: 56 人
shape: (10, 2)
┌───────────────────────┬──────────────────────────────┐
│ celebrity_name        ┆ celebrity_homecountry/region │
│ ---                   ┆ ---                          │
│ str                   ┆ str                          │
╞═══════════════════════╪══════════════════════════════╡
│ Rachel Hunter         ┆ New Zealand                  │
│ Giselle Fernandez     ┆ Mexico                       │
│ Jerry Springer        ┆ England                      │
│ Paulina Porizkova     ┆ Czechoslovakia               │
│ Heather Mills         ┆ England                      │
│ Cameron Mathison      ┆ Canada                       │
│ Jane Seymour          ┆ England                      │
│ Helio Castroneves     ┆ Brazil                       │
│ Mel B                 ┆ England                      │
│ Cristian 

### 2.2 重复值检查

检查是否存在重复记录。注意：同一选手可能参加多季（如全明星季），这不算重复。

In [9]:
# 检查完全重复的行
n_duplicates = df.shape[0] - df.unique().shape[0]
print(f"🔄 完全重复行数: {n_duplicates}")

# 检查同一选手是否参加多季
multi_season = (
    df.group_by("celebrity_name")
    .agg([
        pl.len().alias("appearances"),
        pl.col("season").alias("seasons")
    ])
    .filter(pl.col("appearances") > 1)
    .sort("appearances", descending=True)
)
print(f"\n🔁 参加多季的选手: {multi_season.shape[0]} 人")
if multi_season.shape[0] > 0:
    print(multi_season.head(10))

🔄 完全重复行数: 0

🔁 参加多季的选手: 13 人
shape: (10, 3)
┌───────────────────┬─────────────┬───────────┐
│ celebrity_name    ┆ appearances ┆ seasons   │
│ ---               ┆ ---         ┆ ---       │
│ str               ┆ u32         ┆ list[i64] │
╞═══════════════════╪═════════════╪═══════════╡
│ Kirstie Alley     ┆ 2           ┆ [12, 15]  │
│ Emmitt Smith      ┆ 2           ┆ [3, 15]   │
│ Joey Fatone       ┆ 2           ┆ [4, 15]   │
│ Kelly Monaco      ┆ 2           ┆ [1, 15]   │
│ Bristol Palin     ┆ 2           ┆ [11, 15]  │
│ Drew Lachey       ┆ 2           ┆ [2, 15]   │
│ Apolo Anton Ohno  ┆ 2           ┆ [4, 15]   │
│ Helio Castroneves ┆ 2           ┆ [5, 15]   │
│ Melissa Rycroft   ┆ 2           ┆ [8, 15]   │
│ Gilles Marini     ┆ 2           ┆ [8, 15]   │
└───────────────────┴─────────────┴───────────┘


### 2.3 数据类型转换

确保所有列的数据类型正确，特别是数值型评分列。

In [10]:
# 获取所有评分列
score_cols = [col for col in df.columns if "judge" in col and "score" in col]
print(f"📊 评分列数量: {len(score_cols)}")

# 检查评分列的数据类型
print("\n当前评分列数据类型:")
score_types = {col: df.schema[col] for col in score_cols[:8]}
for col, dtype in score_types.items():
    print(f"  {col}: {dtype}")

📊 评分列数量: 44

当前评分列数据类型:
  week1_judge1_score: Float64
  week1_judge2_score: Float64
  week1_judge3_score: Float64
  week1_judge4_score: Int64
  week2_judge1_score: Float64
  week2_judge2_score: Float64
  week2_judge3_score: Float64
  week2_judge4_score: Float64


In [11]:
# 确保评分列为浮点数类型（处理小数分数）
df = df.with_columns([
    pl.col(col).cast(pl.Float64) for col in score_cols
])

# 确保 season 和 placement 为整数
df = df.with_columns([
    pl.col("season").cast(pl.Int32),
    pl.col("placement").cast(pl.Int32),
    pl.col("celebrity_age_during_season").cast(pl.Int32)
])

print("✅ 数据类型转换完成")
print(f"   - 评分列: Float64")
print(f"   - season/placement/age: Int32")

✅ 数据类型转换完成
   - 评分列: Float64
   - season/placement/age: Int32


### 2.4 异常值检测

评委打分范围应为 1-10（题目说明）。检查是否有超出范围的异常值。

**注意**：
- 0分表示选手已淘汰，是合理值
- 存在bonus分数导致超过10分的情况

In [12]:
# 检查评分范围
print("📊 评分范围检查:")

# 统计所有评分的范围
all_scores = []
for col in score_cols:
    scores = df.select(pl.col(col)).to_series().drop_nulls().to_list()
    all_scores.extend(scores)

all_scores = np.array(all_scores)
print(f"  总评分数量: {len(all_scores)}")
print(f"  最小值: {all_scores.min():.2f}")
print(f"  最大值: {all_scores.max():.2f}")
print(f"  均值: {all_scores.mean():.2f}")

# 检查异常高分（>10）
high_scores = all_scores[all_scores > 10]
print(f"\n⚠️ 超过10分的评分数量: {len(high_scores)}")
if len(high_scores) > 0:
    print(f"   这些通常是bonus分数，范围: {high_scores.min():.2f} - {high_scores.max():.2f}")

📊 评分范围检查:
  总评分数量: 13783
  最小值: 0.00
  最大值: 13.33
  均值: 5.24

⚠️ 超过10分的评分数量: 221
   这些通常是bonus分数，范围: 10.12 - 13.33


In [13]:
# 找出有bonus分数的具体记录
print("🎁 含bonus分数的选手示例:")

# 检查第一周是否有超过10分的情况
for week in range(1, 5):
    for judge in range(1, 5):
        col = f"week{week}_judge{judge}_score"
        if col in df.columns:
            high = df.filter(pl.col(col) > 10)
            if high.shape[0] > 0:
                print(f"\n{col}:")
                print(high.select(["celebrity_name", "season", col]).head(3))

🎁 含bonus分数的选手示例:

week1_judge1_score:
shape: (3, 3)
┌────────────────┬────────┬────────────────────┐
│ celebrity_name ┆ season ┆ week1_judge1_score │
│ ---            ┆ ---    ┆ ---                │
│ str            ┆ i32    ┆ f64                │
╞════════════════╪════════╪════════════════════╡
│ Joanna Krupa   ┆ 9      ┆ 11.3333            │
│ Aaron Carter   ┆ 9      ┆ 10.3333            │
│ Mya            ┆ 9      ┆ 11.3333            │
└────────────────┴────────┴────────────────────┘

week1_judge2_score:
shape: (3, 3)
┌────────────────┬────────┬────────────────────┐
│ celebrity_name ┆ season ┆ week1_judge2_score │
│ ---            ┆ ---    ┆ ---                │
│ str            ┆ i32    ┆ f64                │
╞════════════════╪════════╪════════════════════╡
│ Joanna Krupa   ┆ 9      ┆ 11.3333            │
│ Aaron Carter   ┆ 9      ┆ 11.3333            │
│ Kelly Osbourne ┆ 9      ┆ 10.6666            │
└────────────────┴────────┴────────────────────┘

week1_judge3_score:
shape: (3,

### 2.5 特殊情况处理

处理 "Withdrew"（退赛）选手，标记便于后续分析排除。

In [14]:
# 识别退赛选手
withdrew = df.filter(pl.col("results").str.contains("Withdrew"))
print(f"🚪 退赛选手数量: {withdrew.shape[0]}")
print(withdrew.select(["celebrity_name", "season", "results", "placement"]))

# 添加退赛标记列
df = df.with_columns(
    pl.col("results").str.contains("Withdrew").alias("is_withdrew")
)
print("\n✅ 已添加 is_withdrew 标记列")

🚪 退赛选手数量: 10
shape: (10, 4)
┌──────────────────────┬────────┬──────────┬───────────┐
│ celebrity_name       ┆ season ┆ results  ┆ placement │
│ ---                  ┆ ---    ┆ ---      ┆ ---       │
│ str                  ┆ i32    ┆ str      ┆ i32       │
╞══════════════════════╪════════╪══════════╪═══════════╡
│ Sara Evans           ┆ 3      ┆ Withdrew ┆ 6         │
│ Misty May-Treanor    ┆ 7      ┆ Withdrew ┆ 10        │
│ Tom DeLay            ┆ 9      ┆ Withdrew ┆ 13        │
│ Dorothy Hamill       ┆ 16     ┆ Withdrew ┆ 12        │
│ Billy Dee Williams   ┆ 18     ┆ Withdrew ┆ 10        │
│ Tamar Braxton        ┆ 21     ┆ Withdrew ┆ 5         │
│ Kim Zolciak-Biermann ┆ 21     ┆ Withdrew ┆ 11        │
│ Ray Lewis            ┆ 28     ┆ Withdrew ┆ 11        │
│ Jeannie Mai          ┆ 29     ┆ Withdrew ┆ 9         │
│ Selma Blair          ┆ 31     ┆ Withdrew ┆ 12        │
└──────────────────────┴────────┴──────────┴───────────┘

✅ 已添加 is_withdrew 标记列


---
## 第三步：数据结构转换

原始数据为宽表格式（每周每评委一列），为便于分析，我们创建两个版本：
1. **宽表**：保留原结构，便于选手级别分析
2. **长表**：每行一个「选手-周次」记录，便于时序分析

### 3.1 提取选手基础信息表

将选手基础信息与周评分分离，便于后续关联分析。

In [15]:
# 基础信息列
info_cols = [
    "celebrity_name", "ballroom_partner", "celebrity_industry",
    "celebrity_homestate", "celebrity_homecountry/region",
    "celebrity_age_during_season", "season", "results", "placement", "is_withdrew"
]

# 创建选手信息表
df_contestants = df.select(info_cols)

# 添加唯一标识符（选手名+季数）
df_contestants = df_contestants.with_columns(
    (pl.col("celebrity_name") + "_S" + pl.col("season").cast(pl.Utf8)).alias("contestant_id")
)

print(f"📋 选手信息表维度: {df_contestants.shape}")
df_contestants.head(5)

📋 选手信息表维度: (421, 11)


celebrity_name,ballroom_partner,celebrity_industry,celebrity_homestate,celebrity_homecountry/region,celebrity_age_during_season,season,results,placement,is_withdrew,contestant_id
str,str,str,str,str,i32,i32,str,i32,bool,str
"""John O'Hurley""","""Charlotte Jorgensen""","""Actor/Actress""","""Maine""","""United States""",50,1,"""2nd Place""",2,false,"""John O'Hurley_S1"""
"""Kelly Monaco""","""Alec Mazo""","""Actor/Actress""","""Pennsylvania""","""United States""",29,1,"""1st Place""",1,false,"""Kelly Monaco_S1"""
"""Evander Holyfield""","""Edyta Sliwinska""","""Athlete""","""Alabama""","""United States""",42,1,"""Eliminated Week 3""",5,false,"""Evander Holyfield_S1"""
"""Rachel Hunter""","""Jonathan Roberts""","""Model""",null,"""New Zealand""",35,1,"""Eliminated Week 4""",4,false,"""Rachel Hunter_S1"""
"""Joey McIntyre""","""Ashly DelGrosso""","""Singer/Rapper""","""Massachusetts""","""United States""",32,1,"""3rd Place""",3,false,"""Joey McIntyre_S1"""


### 3.2 创建周评分长表

将宽表转换为长表格式：每行代表一个「选手 × 周次 × 评委」的评分记录。

**目的**：便于进行时序分析和评委间比较。

In [16]:
# 准备转换：提取所有周次和评委组合
weeks = list(range(1, 12))  # week1 ~ week11
judges = list(range(1, 5))   # judge1 ~ judge4

# 构建长表
records = []

for row in df.iter_rows(named=True):
    contestant_id = f"{row['celebrity_name']}_S{row['season']}"
    
    for week in weeks:
        week_scores = []
        for judge in judges:
            col = f"week{week}_judge{judge}_score"
            score = row.get(col)
            
            if score is not None:
                records.append({
                    "contestant_id": contestant_id,
                    "celebrity_name": row["celebrity_name"],
                    "season": row["season"],
                    "week": week,
                    "judge": judge,
                    "score": score
                })

df_scores_long = pl.DataFrame(records)
print(f"📊 周评分长表维度: {df_scores_long.shape}")
df_scores_long.head(10)

📊 周评分长表维度: (13783, 6)


contestant_id,celebrity_name,season,week,judge,score
str,str,i64,i64,i64,f64
"""John O'Hurley_S1""","""John O'Hurley""",1,1,1,7.0
"""John O'Hurley_S1""","""John O'Hurley""",1,1,2,7.0
"""John O'Hurley_S1""","""John O'Hurley""",1,1,3,6.0
"""John O'Hurley_S1""","""John O'Hurley""",1,2,1,8.0
"""John O'Hurley_S1""","""John O'Hurley""",1,2,2,9.0
"""John O'Hurley_S1""","""John O'Hurley""",1,2,3,9.0
"""John O'Hurley_S1""","""John O'Hurley""",1,3,1,9.0
"""John O'Hurley_S1""","""John O'Hurley""",1,3,2,8.0
"""John O'Hurley_S1""","""John O'Hurley""",1,3,3,7.0


### 3.3 创建周汇总表

计算每个选手每周的总分、平均分等汇总指标。

In [17]:
# 过滤掉0分（淘汰后）的记录进行汇总
df_week_summary = (
    df_scores_long
    .filter(pl.col("score") > 0)  # 排除淘汰后的0分
    .group_by(["contestant_id", "celebrity_name", "season", "week"])
    .agg([
        pl.sum("score").alias("total_score"),
        pl.mean("score").alias("avg_score"),
        pl.count("score").alias("judge_count"),
        pl.min("score").alias("min_score"),
        pl.max("score").alias("max_score")
    ])
    .sort(["season", "week", "total_score"], descending=[False, False, True])
)

print(f"📊 周汇总表维度: {df_week_summary.shape}")
df_week_summary.head(10)

📊 周汇总表维度: (2777, 9)


contestant_id,celebrity_name,season,week,total_score,avg_score,judge_count,min_score,max_score
str,str,i64,i64,f64,f64,u32,f64,f64
"""Rachel Hunter_S1""","""Rachel Hunter""",1,1,20.0,6.666667,3,6.0,7.0
"""John O'Hurley_S1""","""John O'Hurley""",1,1,20.0,6.666667,3,6.0,7.0
"""Joey McIntyre_S1""","""Joey McIntyre""",1,1,20.0,6.666667,3,6.0,7.0
"""Evander Holyfield_S1""","""Evander Holyfield""",1,1,18.0,6.0,3,5.0,7.0
"""Trista Sutter_S1""","""Trista Sutter""",1,1,18.0,6.0,3,6.0,6.0
"""Kelly Monaco_S1""","""Kelly Monaco""",1,1,13.0,4.333333,3,4.0,5.0
"""John O'Hurley_S1""","""John O'Hurley""",1,2,26.0,8.666667,3,8.0,9.0
"""Rachel Hunter_S1""","""Rachel Hunter""",1,2,24.0,8.0,3,8.0,8.0
"""Joey McIntyre_S1""","""Joey McIntyre""",1,2,21.0,7.0,3,6.0,8.0


In [18]:
# 计算每周排名（用于分析投票方法）
df_week_summary = df_week_summary.with_columns(
    pl.col("total_score")
    .rank(method="ordinal", descending=True)
    .over(["season", "week"])
    .alias("week_rank")
)

print("✅ 已添加周排名列")
df_week_summary.filter((pl.col("season") == 1) & (pl.col("week") == 4))

✅ 已添加周排名列


contestant_id,celebrity_name,season,week,total_score,avg_score,judge_count,min_score,max_score,week_rank
str,str,i64,i64,f64,f64,u32,f64,f64,u32
"""Kelly Monaco_S1""","""Kelly Monaco""",1,4,26.0,8.666667,3,8.0,9.0,1
"""Rachel Hunter_S1""","""Rachel Hunter""",1,4,25.0,8.333333,3,7.0,9.0,2
"""John O'Hurley_S1""","""John O'Hurley""",1,4,21.0,7.0,3,6.0,8.0,3
"""Joey McIntyre_S1""","""Joey McIntyre""",1,4,20.0,6.666667,3,6.0,7.0,4


---
## 第四步：特征工程

根据题目分析需求，创建以下特征类型。

### 4.1 选手累计表现特征

计算选手在整季的累计表现指标。

In [19]:
# 计算选手整季汇总指标
df_season_stats = (
    df_week_summary
    .group_by(["contestant_id", "celebrity_name", "season"])
    .agg([
        pl.len().alias("weeks_competed"),
        pl.sum("total_score").alias("season_total_score"),
        pl.mean("total_score").alias("season_avg_weekly_score"),
        pl.mean("avg_score").alias("season_avg_judge_score"),
        pl.std("total_score").alias("score_volatility"),
        pl.mean("week_rank").alias("avg_rank"),
        (pl.col("week_rank") == 1).sum().alias("weeks_ranked_first")
    ])
)

# 合并选手基础信息
df_season_stats = df_season_stats.join(
    df_contestants.select(["contestant_id", "placement", "results", "ballroom_partner", 
                           "celebrity_industry", "celebrity_age_during_season", "is_withdrew"]),
    on="contestant_id",
    how="left"
)

print(f"📊 选手季度统计表维度: {df_season_stats.shape}")
df_season_stats.head(10)

📊 选手季度统计表维度: (421, 16)


contestant_id,celebrity_name,season,weeks_competed,season_total_score,season_avg_weekly_score,season_avg_judge_score,score_volatility,avg_rank,weeks_ranked_first,placement,results,ballroom_partner,celebrity_industry,celebrity_age_during_season,is_withdrew
str,str,i64,u32,f64,f64,f64,f64,f64,u32,i32,str,str,str,i32,bool
"""Sugar Ray Leonard_S12""","""Sugar Ray Leonard""",12,4,75.0,18.75,6.25,2.061553,9.0,0,9,"""Eliminated Week 4""","""Anna Trebunskaya""","""Athlete""",54,false
"""Xochitl Gomez_S32""","""Xochitl Gomez""",32,11,349.0,31.727273,9.121212,7.603827,2.0,5,1,"""1st Place""","""Valentin Chmerkovskiy""","""Actor/Actress""",17,false
"""Vanilla Ice_S23""","""Vanilla Ice""",23,4,97.0,24.25,6.541667,1.5,8.0,0,10,"""Eliminated Week 4""","""Witney Carson""","""Singer/Rapper""",48,false
"""Billy Dee Williams_S18""","""Billy Dee Williams""",18,2,30.0,15.0,5.0,0.0,11.5,0,10,"""Withdrew""","""Emma Slater""","""Actor/Actress""",76,true
"""Donny Osmond_S9""","""Donny Osmond""",9,10,264.6662,26.46662,8.822207,3.155201,3.5,2,1,"""1st Place""","""Kym Johnson""","""Singer/Rapper""",51,false
"""The Situation_S11""","""The Situation""",11,4,67.0,16.75,5.583333,2.753785,9.0,0,9,"""Eliminated Week 4""","""Karina Smirnoff""","""TV Personality""",28,false
"""Pamela Anderson_S15""","""Pamela Anderson""",15,1,17.0,17.0,5.666667,null,13.0,0,13,"""Eliminated Week 1""","""Tristan MacManus""","""Model""",45,false
"""Justina Machado_S29""","""Justina Machado""",29,11,275.9996,25.090873,8.363624,3.645643,4.454545,1,4,"""4th Place""","""Sasha Farber""","""Actor/Actress""",48,false
"""Johnny Weir_S29""","""Johnny Weir""",29,10,250.5,25.05,8.35,4.609471,4.1,1,6,"""Eliminated Week 10""","""Britt Stewart""","""Athlete""",36,false


### 4.2 时序特征：逐周表现趋势

计算选手的周环比变化，用于分析成长曲线。

In [21]:
# 计算周环比变化
df_week_trend = (
    df_week_summary
    .sort(["contestant_id", "week"])
    .with_columns([
        # 上周分数
        pl.col("total_score").shift(1).over("contestant_id").alias("prev_week_score"),
        # 累计平均分（使用 cum_sum / row_nr 实现）
        (pl.col("total_score").cum_sum().over("contestant_id") / 
         (pl.int_range(1, pl.len() + 1).over("contestant_id"))).alias("cumulative_avg"),
    ])
    .with_columns([
        # 周环比变化
        (pl.col("total_score") - pl.col("prev_week_score")).alias("score_change"),
        # 相对累计平均的偏差
        (pl.col("total_score") - pl.col("cumulative_avg")).alias("deviation_from_avg")
    ])
)

print(f"📊 周趋势表维度: {df_week_trend.shape}")
df_week_trend.filter(pl.col("contestant_id") == "Kelly Monaco_S1").head(10)

📊 周趋势表维度: (2777, 14)


contestant_id,celebrity_name,season,week,total_score,avg_score,judge_count,min_score,max_score,week_rank,prev_week_score,cumulative_avg,score_change,deviation_from_avg
str,str,i64,i64,f64,f64,u32,f64,f64,u32,f64,f64,f64,f64
"""Kelly Monaco_S1""","""Kelly Monaco""",1,1,13.0,4.333333,3,4.0,5.0,6,null,13.0,null,0.0
"""Kelly Monaco_S1""","""Kelly Monaco""",1,2,17.0,5.666667,3,5.0,6.0,5,13.0,15.0,4.0,2.0
"""Kelly Monaco_S1""","""Kelly Monaco""",1,3,21.0,7.0,3,6.0,8.0,4,17.0,17.0,4.0,4.0
"""Kelly Monaco_S1""","""Kelly Monaco""",1,4,26.0,8.666667,3,8.0,9.0,1,21.0,19.25,5.0,6.75
"""Kelly Monaco_S1""","""Kelly Monaco""",1,5,23.5,7.833333,3,7.5,8.5,2,26.0,20.1,-2.5,3.4
"""Kelly Monaco_S1""","""Kelly Monaco""",1,6,27.5,9.166667,3,8.5,9.5,1,23.5,21.333333,4.0,6.166667


### 4.3 分类编码

对分类变量进行编码，便于后续建模。

In [22]:
# 职业类别编码
industries = df_contestants["celebrity_industry"].unique().sort().to_list()
industry_mapping = {ind: i for i, ind in enumerate(industries)}

print("🏷️ 职业类别编码:")
for ind, code in industry_mapping.items():
    print(f"  {code}: {ind}")

🏷️ 职业类别编码:
  0: Actor/Actress
  1: Astronaut
  2: Athlete
  3: Beauty Pagent
  4: Comedian
  5: Con artist
  6: Conservationist
  7: Entrepreneur
  8: Fashion Designer
  9: Fitness Instructor
  10: Journalist
  11: Magician
  12: Military
  13: Model
  14: Motivational Speaker
  15: Musician
  16: News Anchor
  17: Politician
  18: Producer
  19: Racing Driver
  20: Radio Personality
  21: Singer/Rapper
  22: Social Media Personality
  23: Social media personality
  24: Sports Broadcaster
  25: TV Personality


In [23]:
# 添加编码列到季度统计表
df_season_stats = df_season_stats.with_columns(
    pl.col("celebrity_industry").replace(industry_mapping).alias("industry_code")
)

# 添加投票方法标记（根据题目说明）
# 季1-2: 排名法, 季3-27: 百分比法, 季28-34: 排名法（含底二淘汰机制）
df_season_stats = df_season_stats.with_columns(
    pl.when(pl.col("season") <= 2)
    .then(pl.lit("rank_v1"))
    .when(pl.col("season") <= 27)
    .then(pl.lit("percentage"))
    .otherwise(pl.lit("rank_v2"))
    .alias("voting_method")
)

print("✅ 已添加编码列:")
print("   - industry_code: 职业类别编码")
print("   - voting_method: 投票方法标记")

✅ 已添加编码列:
   - industry_code: 职业类别编码
   - voting_method: 投票方法标记


### 4.4 舞伴统计特征

分析专业舞伴的「带人能力」——不同舞伴带出的选手平均排名。

In [24]:
# 计算每位舞伴的历史表现
df_partner_stats = (
    df_season_stats
    .filter(~pl.col("is_withdrew"))  # 排除退赛
    .group_by("ballroom_partner")
    .agg([
        pl.len().alias("total_partners"),
        pl.mean("placement").alias("avg_placement"),
        pl.min("placement").alias("best_placement"),
        (pl.col("placement") == 1).sum().alias("championships"),
        (pl.col("placement") <= 3).sum().alias("top3_finishes"),
        pl.mean("season_avg_judge_score").alias("avg_judge_score")
    ])
    .filter(pl.col("total_partners") >= 3)  # 至少3次合作
    .sort("avg_placement")
)

print("💃 舞伴表现排行（按平均名次）:")
print(df_partner_stats.head(15))

💃 舞伴表现排行（按平均名次）:
shape: (15, 7)
┌──────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ ballroom_par ┆ total_partn ┆ avg_placeme ┆ best_placem ┆ championshi ┆ top3_finish ┆ avg_judge_s │
│ tner         ┆ ers         ┆ nt          ┆ ent         ┆ ps          ┆ es          ┆ core        │
│ ---          ┆ ---         ┆ ---         ┆ ---         ┆ ---         ┆ ---         ┆ ---         │
│ str          ┆ u32         ┆ f64         ┆ i32         ┆ u32         ┆ u32         ┆ f64         │
╞══════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ Derek Hough  ┆ 17          ┆ 2.941176    ┆ 1           ┆ 6           ┆ 9           ┆ 8.881499    │
│ Julianne     ┆ 5           ┆ 4.2         ┆ 1           ┆ 2           ┆ 2           ┆ 7.85361     │
│ Hough        ┆             ┆             ┆             ┆             ┆             ┆             │
│ Daniella     ┆ 5           ┆ 4.6         ┆ 1           ┆ 

### 4.5 争议案例标记

根据题目提及的争议案例，创建标记便于专项分析。

In [25]:
# 定义争议案例
controversy_cases = [
    ("Jerry Rice", 2, "亚军但5周最低评委分"),
    ("Billy Ray Cyrus", 4, "第5名但6周最低评委分"),
    ("Bristol Palin", 11, "季军但12次最低评委分"),
    ("Bobby Bones", 27, "冠军但持续低评委分"),
]

# 创建争议案例标记
controversy_ids = [f"{name}_S{season}" for name, season, _ in controversy_cases]

df_season_stats = df_season_stats.with_columns(
    pl.col("contestant_id").is_in(controversy_ids).alias("is_controversy_case")
)

print("⚠️ 争议案例:")
print(df_season_stats.filter(pl.col("is_controversy_case")).select(
    ["celebrity_name", "season", "placement", "season_avg_judge_score", "avg_rank"]
))

⚠️ 争议案例:
shape: (4, 5)
┌─────────────────┬────────┬───────────┬────────────────────────┬──────────┐
│ celebrity_name  ┆ season ┆ placement ┆ season_avg_judge_score ┆ avg_rank │
│ ---             ┆ ---    ┆ ---       ┆ ---                    ┆ ---      │
│ str             ┆ i64    ┆ i32       ┆ f64                    ┆ f64      │
╞═════════════════╪════════╪═══════════╪════════════════════════╪══════════╡
│ Jerry Rice      ┆ 2      ┆ 2         ┆ 7.506942               ┆ 4.625    │
│ Bristol Palin   ┆ 11     ┆ 3         ┆ 7.368047               ┆ 5.9      │
│ Billy Ray Cyrus ┆ 4      ┆ 5         ┆ 6.333333               ┆ 7.0      │
│ Bobby Bones     ┆ 27     ┆ 1         ┆ 7.462963               ┆ 7.222222 │
└─────────────────┴────────┴───────────┴────────────────────────┴──────────┘


---
## 第五步：数据验证与保存

验证处理后数据的完整性，并保存为多个文件供后续建模使用。

### 5.1 数据完整性检查

In [26]:
print("=" * 60)
print("🔍 数据完整性检查")
print("=" * 60)

# 检查各表记录数
print(f"\n📊 数据表概览:")
print(f"  - 原始数据: {df_raw.shape[0]} 行")
print(f"  - 选手信息表: {df_contestants.shape[0]} 行")
print(f"  - 周评分长表: {df_scores_long.shape[0]} 行")
print(f"  - 周汇总表: {df_week_summary.shape[0]} 行")
print(f"  - 季度统计表: {df_season_stats.shape[0]} 行")
print(f"  - 舞伴统计表: {df_partner_stats.shape[0]} 行")

# 验证季数覆盖
seasons_in_data = df_season_stats["season"].unique().sort().to_list()
print(f"\n📅 覆盖季数: {min(seasons_in_data)} ~ {max(seasons_in_data)} (共 {len(seasons_in_data)} 季)")

# 检查关键字段缺失
print(f"\n🔑 关键字段缺失检查 (季度统计表):")
for col in ["placement", "season_avg_judge_score", "weeks_competed"]:
    null_count = df_season_stats.filter(pl.col(col).is_null()).shape[0]
    print(f"  {col}: {null_count} 缺失")

🔍 数据完整性检查

📊 数据表概览:
  - 原始数据: 421 行
  - 选手信息表: 421 行
  - 周评分长表: 13783 行
  - 周汇总表: 2777 行
  - 季度统计表: 421 行
  - 舞伴统计表: 36 行

📅 覆盖季数: 1 ~ 34 (共 34 季)

🔑 关键字段缺失检查 (季度统计表):
  placement: 0 缺失
  season_avg_judge_score: 0 缺失
  weeks_competed: 0 缺失


In [30]:
# 交叉验证：检查placement与results的一致性
print("\n🔄 结果一致性检查:")

# 检查冠军placement是否为1
champions = df_season_stats.filter(pl.col("results").str.contains("1st Place"))
champ_placements = champions["placement"].unique().to_list()
print(f"  冠军placement值: {champ_placements} (应为 [1])")

# 检查每季是否有且仅有一个冠军
champs_per_season = champions.group_by("season").agg(pl.len().alias("count"))
multi_champs = champs_per_season.filter(pl.col("count") > 1)
print(f"  多冠军季数: {multi_champs.shape[0]} (应为 0)")


🔄 结果一致性检查:
  冠军placement值: [1] (应为 [1])
  多冠军季数: 0 (应为 0)


### 5.2 保存处理后数据

In [28]:
# 保存各数据表
output_files = {
    "contestants.csv": df_contestants,
    "scores_long.csv": df_scores_long,
    "week_summary.csv": df_week_trend,  # 包含趋势特征的版本
    "season_stats.csv": df_season_stats,
    "partner_stats.csv": df_partner_stats,
}

print("💾 保存处理后数据:")
for filename, data in output_files.items():
    filepath = DATA_PROCESSED / filename
    data.write_csv(filepath)
    print(f"  ✅ {filepath} ({data.shape[0]} 行 × {data.shape[1]} 列)")

print(f"\n📁 所有文件已保存至: {DATA_PROCESSED.absolute()}")

💾 保存处理后数据:
  ✅ data\processed\contestants.csv (421 行 × 11 列)
  ✅ data\processed\scores_long.csv (13783 行 × 6 列)
  ✅ data\processed\week_summary.csv (2777 行 × 14 列)
  ✅ data\processed\season_stats.csv (421 行 × 19 列)
  ✅ data\processed\partner_stats.csv (36 行 × 7 列)

📁 所有文件已保存至: d:\Download\mcm\mcm\data\processed


### 5.3 处理流程总结

| 步骤 | 操作 | 输出 |
|------|------|------|
| 1. 数据加载 | 读取CSV，处理N/A值 | `df_raw` |
| 2. 数据清洗 | 类型转换、退赛标记 | `df` (清洗后) |
| 3. 结构转换 | 宽表→长表 | `df_scores_long`, `df_week_summary` |
| 4. 特征工程 | 累计指标、趋势、编码 | `df_season_stats`, `df_partner_stats` |
| 5. 保存输出 | CSV文件 | `data/processed/` 目录 |

### 关键决策记录

1. **N/A值保留**：不删除或填充，因其有明确业务含义
2. **0分过滤**：在汇总计算时排除（淘汰后的0分）
3. **超10分保留**：bonus分数是真实数据，不做截断
4. **投票方法分期**：S1-2为rank_v1，S3-27为percentage，S28+为rank_v2
5. **舞伴筛选**：至少3次合作才纳入统计，避免小样本偏差

In [29]:
# 最终数据预览
print("📊 处理后数据预览 - 季度统计表:")
df_season_stats.select([
    "celebrity_name", "season", "placement", "weeks_competed",
    "season_avg_judge_score", "avg_rank", "voting_method", "is_controversy_case"
]).head(15)

📊 处理后数据预览 - 季度统计表:


celebrity_name,season,placement,weeks_competed,season_avg_judge_score,avg_rank,voting_method,is_controversy_case
str,i64,i32,u32,f64,f64,str,bool
"""Sugar Ray Leonard""",12,9,4,6.25,9.0,"""percentage""",false
"""Xochitl Gomez""",32,1,11,9.121212,2.0,"""rank_v2""",false
"""Vanilla Ice""",23,10,4,6.541667,8.0,"""percentage""",false
"""Billy Dee Williams""",18,10,2,5.0,11.5,"""percentage""",false
"""Donny Osmond""",9,1,10,8.822207,3.5,"""percentage""",false
"""The Situation""",11,9,4,5.583333,9.0,"""percentage""",false
"""Pamela Anderson""",15,13,1,5.666667,13.0,"""percentage""",false
"""Justina Machado""",29,4,11,8.363624,4.454545,"""rank_v2""",false
"""Johnny Weir""",29,6,10,8.35,4.1,"""rank_v2""",false


---
## 下一步建议

基于预处理后的数据，后续分析可按以下方向展开：

1. **观众投票估算模型**
   - 使用 `week_summary.csv` 的排名数据
   - 基于淘汰约束反推投票分布

2. **投票方法对比分析**
   - 使用 `season_stats.csv` 的 `voting_method` 字段分组
   - 模拟不同方法下的排名变化

3. **争议案例深入分析**
   - 筛选 `is_controversy_case=True` 的记录
   - 分析评委分与最终排名的背离程度

4. **舞伴/选手特征影响分析**
   - 使用 `partner_stats.csv` 和 `season_stats.csv`
   - 回归分析各因素对排名的影响